# Chicago Closure Radar — Step 4: Model Training & Results

Train the XGBoost closure-risk classifier and produce:
- ROC-AUC and Average Precision scores
- SHAP feature importance
- Risk score leaderboard for current Chicago businesses
- Validation: did we flag the real closures early enough?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

from src.models.closure_risk import prepare_xy, train_xgb, evaluate, compute_risk_scores

df = pd.read_parquet('../data/processed/features_combined.parquet')
print(f'Rows: {len(df):,} | Label=1: {df["label"].sum():,} ({100*df["label"].mean():.1f}%)')

## Train / Test Split & Model Training

In [ ]:
X, y = prepare_xy(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

clf = train_xgb(X_train, y_train)
metrics = evaluate(clf, X_test, y_test)
print(metrics)

## ROC + Precision-Recall Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_estimator(clf, X_test, y_test, ax=ax1)
ax1.set_title('ROC Curve — Closure Risk Model')

PrecisionRecallDisplay.from_estimator(clf, X_test, y_test, ax=ax2)
ax2.set_title('Precision-Recall Curve')

plt.tight_layout()
plt.savefig('../outputs/figures/roc_pr_curves.png', dpi=150)
plt.show()

## SHAP Feature Importance

In [ ]:
explainer = shap.TreeExplainer(clf)
shap_vals = explainer.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, X_test, plot_type='bar', show=False, max_display=20)
plt.title('Top Features by SHAP Importance')
plt.tight_layout()
plt.savefig('../outputs/figures/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Risk Score Leaderboard — Highest-Risk Current Businesses

In [ ]:
biz = pd.read_parquet('../data/processed/yelp_chicago_businesses.parquet')
scores = compute_risk_scores(clf, X, df)

# Show top at-risk open businesses
at_risk = scores[scores['risk_bucket'] == 'high'].sort_values('risk_score', ascending=False)
print(f'HIGH RISK businesses: {len(at_risk)}')
at_risk[['name', 'categories', 'risk_score', 'risk_bucket']].head(20)

## Validation: How Early Did We Flag Real Closures?

The pitch: **flagged 6 months before they closed**.

In [ ]:
gt = pd.read_parquet('../data/ground_truth/ground_truth.parquet')

# Join predictions to ground truth
validation = scores.merge(
    gt[['business_id', 'is_closed', 'closure_date']], on='business_id', how='left'
)
validation = validation[validation['is_closed'] == True]

# Risk score distribution for actually-closed businesses
fig, ax = plt.subplots(figsize=(8, 4))
validation['risk_score'].hist(bins=30, ax=ax, color='tomato', edgecolor='white', alpha=0.8)
ax.axvline(0.66, color='black', linestyle='--', label='High-risk threshold')
ax.set_xlabel('Risk Score')
ax.set_ylabel('Count')
ax.set_title('Risk Score Distribution for Businesses That Actually Closed')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/risk_score_closed_distribution.png', dpi=150)
plt.show()

pct_flagged = (validation['risk_score'] >= 0.66).mean()
print(f'\n>>> {pct_flagged:.1%} of eventually-closed businesses were flagged HIGH RISK')